# SASV: ECAPA + WavLM locked **eval** (report once)

Official SASV **eval** with your app **WavLM** CM:

```text
s_sasv = s_asv + (1 - P_spoof)
```

**Rules**

- Finish `05_ecapa_plus_wavlm_dev.ipynb` (full **dev**) first.
- Do **not** change fusion / thresholds after you see these eval numbers.
- ECAPA-only eval is skipped by default (already in `04`); set `RUN_ECAPA_ONLY = True` only if missing.

WavLM is slower than LFCC on ~102k trials — prefer CUDA.

Needs `transformers` in the `app/server` kernel.

In [1]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if not (ROOT / "score_lib.py").exists():
    ROOT = Path(r"D:\speaker-verification-system\replay-cnn-baseline\experiments\sasv_la2019")
sys.path.insert(0, str(ROOT))

import torch
from experiment_lib import DEFAULT_LA, DEFAULT_SASV, RUNS_DIR
from score_lib import score_ecapa_trials, score_fused_trials

print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
print("LA exists:", DEFAULT_LA.exists())
print("SASV exists:", DEFAULT_SASV.exists())

cuda: True
NVIDIA GeForce RTX 4060 Laptop GPU
LA exists: True
SASV exists: True


## Locked settings

- `SPLIT` fixed to **`eval`**
- `CM_BACKEND` fixed to **`wavlm`**
- `MAX_TRIALS = 0` → all eval trials

In [2]:
SPLIT = "eval"
MAX_TRIALS = 0
CM_BACKEND = "wavlm"
DEVICE = "cuda"
FORCE_CPU = False

RUN_ECAPA_ONLY = False  # True only if runs/ecapa_only_eval is missing
RUN_FUSED = True

assert SPLIT == "eval"
assert CM_BACKEND == "wavlm"

## 1. ECAPA-only (optional)

In [3]:
if RUN_ECAPA_ONLY:
    ecapa_summary = score_ecapa_trials(
        la_root=DEFAULT_LA,
        sasv_root=DEFAULT_SASV,
        split=SPLIT,
        max_trials=MAX_TRIALS,
        device=DEVICE,
        force_cpu=FORCE_CPU,
        output_dir=RUNS_DIR / f"ecapa_only_{SPLIT}",
    )
    display({
        "system": ecapa_summary["system"],
        "sasv_eer_%": ecapa_summary["sasv_eer_percent"],
        "sv_eer_%": ecapa_summary["sv_eer_percent"],
        "spf_eer_%": ecapa_summary["spf_eer_percent"],
        "n": ecapa_summary["num_scored"],
    })
else:
    print("Skipped ECAPA-only (use runs/ecapa_only_eval from notebook 04)")

Skipped ECAPA-only (use runs/ecapa_only_eval from notebook 04)


## 2. ECAPA + WavLM fusion (eval)

Writes `runs/ecapa_plus_wavlm_eval/`

In [ ]:
fused_summary = None
if RUN_FUSED:
    fused_summary = score_fused_trials(
        la_root=DEFAULT_LA,
        sasv_root=DEFAULT_SASV,
        split=SPLIT,
        max_trials=MAX_TRIALS,
        device=DEVICE,
        force_cpu=FORCE_CPU,
        cm_backend=CM_BACKEND,
        output_dir=RUNS_DIR / f"ecapa_plus_{CM_BACKEND}_{SPLIT}",
    )
    display({
        "system": fused_summary["system"],
        "sasv_eer_%": fused_summary["sasv_eer_percent"],
        "sv_eer_%": fused_summary["sv_eer_percent"],
        "spf_eer_%": fused_summary["spf_eer_percent"],
        "n": fused_summary["num_scored"],
    })
else:
    print("Skipped fused")

Trials: {'target': 5370, 'nontarget': 33327, 'spoof': 63882, 'total': 102579} | CM=wavlm
[WARN] webrtc_noise_gain not installed – WebRTC disabled.


Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.


Enrol eval:   0%|          | 0/48 [00:00<?, ?it/s]

Score fused:   0%|          | 0/102579 [00:00<?, ?it/s]

'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /microsoft/wavlm-base/resolve/main/config.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000261CD263350>: Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: c22f7244-2128-4309-9845-e656293e6363)')' thrown while requesting HEAD https://huggingface.co/microsoft/wavlm-base/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /microsoft/wavlm-base/resolve/main/config.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000261CD279E10>: Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: c69cef8c-805f-414f-8c2c-1c0143703035)')' thrown while requesting HEAD https://huggingface.co/microsoft/wavlm-base/resolve/main/config.json
Retrying i

## 3. Report table

Eval numbers + locked **dev** (if present).

In [ ]:
print("Eval:")
for label, path in [
    ("ecapa_only", RUNS_DIR / "ecapa_only_eval" / "metrics_eval.json"),
    ("ecapa_plus_lfcc", RUNS_DIR / "ecapa_plus_lfcc_eval" / "metrics_eval.json"),
    ("ecapa_plus_wavlm", RUNS_DIR / "ecapa_plus_wavlm_eval" / "metrics_eval.json"),
]:
    if not path.exists():
        print("  Missing:", path)
        continue
    m = json.loads(path.read_text(encoding="utf-8"))
    print(
        f"  {label:20s}  SASV={m['sasv_eer_percent']:.4f}%  "
        f"SV={m['sv_eer_percent']:.4f}%  SPF={m['spf_eer_percent']:.4f}%  n={m['num_scored']}"
    )

print("\nDev (reference):")
for label, path in [
    ("ecapa_only", RUNS_DIR / "ecapa_only_dev" / "metrics_dev.json"),
    ("ecapa_plus_lfcc", RUNS_DIR / "ecapa_plus_lfcc_dev" / "metrics_dev.json"),
    ("ecapa_plus_wavlm", RUNS_DIR / "ecapa_plus_wavlm_dev" / "metrics_dev.json"),
]:
    if not path.exists():
        print("  Missing:", path)
        continue
    m = json.loads(path.read_text(encoding="utf-8"))
    print(
        f"  {label:20s}  SASV={m['sasv_eer_percent']:.4f}%  "
        f"SV={m['sv_eer_percent']:.4f}%  SPF={m['spf_eer_percent']:.4f}%"
    )

## Done

Copy eval EERs into your table. Do not re-tune on these numbers.

Reference (published eval): B1-v2 (ECAPA+AASIST) ~**1.71%** SASV-EER.